# ShelfScan — Entrega 1: YOLOv8 Preliminary Training
**Diego Valenzuela | Visión por Computadora**

In [ ]:
# Install deps (run once)
!pip install ultralytics roboflow opencv-python-headless -q

In [ ]:
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 1. Download dataset from Roboflow

In [ ]:
from roboflow import Roboflow

# Replace with your Roboflow API key and project details
RF_API_KEY = 'YOUR_API_KEY'
RF_WORKSPACE = 'YOUR_WORKSPACE'
RF_PROJECT = 'shelfscan'
RF_VERSION = 1

rf = Roboflow(api_key=RF_API_KEY)
project = rf.workspace(RF_WORKSPACE).project(RF_PROJECT)
dataset = project.version(RF_VERSION).download('yolov8')
print(f'Dataset path: {dataset.location}')

## 2. Run augmentation

In [ ]:
import subprocess, sys, os
os.chdir('/content/ShelfScan')  # adjust if needed
subprocess.run([sys.executable, 'scripts/augmentation.py'], check=True)
subprocess.run([sys.executable, 'scripts/split_dataset.py'], check=True)

## 3. Train YOLOv8n — Preliminary Run

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

results = model.train(
    data='data/dataset.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    patience=10,
    project='models',
    name='shelfscan_v1',
    exist_ok=True,
    plots=True,
)

## 4. Evaluate and report mAP

In [ ]:
best_model = YOLO('models/shelfscan_v1/weights/best.pt')
metrics = best_model.val(data='data/dataset.yaml')

print('=== ShelfScan v1 — Preliminary Metrics ===')
print(f'mAP@0.5:      {metrics.box.map50:.4f}')
print(f'mAP@0.5:0.95: {metrics.box.map:.4f}')
print(f'Precision:    {metrics.box.mp:.4f}')
print(f'Recall:       {metrics.box.mr:.4f}')

# Per-class breakdown
from scripts.categories import CLASS_NAMES
print('\nPer-class mAP@0.5:')
for i, (name, ap) in enumerate(zip(CLASS_NAMES, metrics.box.maps)):
    print(f'  {name:15s}: {ap:.4f}')

## 5. Visualize training curves

In [ ]:
from IPython.display import Image
Image('models/shelfscan_v1/results.png')